# RSTC2026 — Optical Satellite Data
## Sentinel-2 vegetation change and land-surface phenology

**Practical notebook — Google Colab**  
**Katarzyna Ostapowicz — Norwegian Institute for Nature Research (NINA)**

This practical uses the **same Adventdalen AOI as Block 1**.

We will work from the scientific question towards the metric, rather than calculate a large set of indices without a clear purpose.

### Workflow

1. Define the Adventdalen study area and two local sampling sites.
2. Load Sentinel-2 Level-2A surface reflectance.
3. Apply pixel-level quality assessment with Cloud Score+ and remove snow/ice pixels.
4. Calculate three complementary indices: NDVI, NDRE and NDMI.
5. Inspect a midsummer composite and check the spatial context of the sample sites.
6. Extract seasonal Sentinel-2 time series for the two local sites.
7. Examine whether the selected indices provide similar or complementary information.
8. Compare midsummer vegetation conditions between years.
9. Map spatial NDVI change across the full AOI.
10. Reconstruct a seasonal NDVI curve and derive simple phenology metrics.

The full AOI is used for **spatial questions**.  
The two local sampling sites are used for **time-series questions**.


## 1. Scientific questions

Before processing the imagery, define what the analysis is intended to measure.

For this exercise we use four questions.

**Q1. Vegetation greenness**  
How does the strength of the green vegetation signal change through the season and between years?

**Q2. Red-edge vegetation response**  
Does Sentinel-2 red-edge information show the same temporal pattern as NDVI?

**Q3. Moisture-related spectral response**  
Does the NIR–SWIR signal vary independently from greenness?

**Q4. Seasonal timing**  
When does the local vegetation signal increase, reach its seasonal maximum and decline?

These questions lead to a small set of metrics:

| Scientific question | Metric | Sentinel-2 bands |
|---|---|---|
| Vegetation greenness | NDVI | B8, B4 |
| Red-edge vegetation response | NDRE | B8A, B5 |
| Moisture-related spectral response | NDMI | B8A, B11 |
| Seasonal timing | SOS, POS, EOS, LOS | derived from the NDVI time series |

We will inspect the satellite observations first and only then interpret the derived metrics.


## 2. Connect Google Colab to Earth Engine

In this step we import the Python libraries used in the practical and connect the notebook to Google Earth Engine.

- `ee` gives Python access to Earth Engine.
- `geemap` displays Earth Engine layers on an interactive map.
- `pandas` stores extracted time-series values in tables.
- `numpy` is used for numerical operations.
- `matplotlib` produces the plots.
- `savgol_filter` is used later to smooth the seasonal NDVI trajectory.

`PROJECT_ID` is the Google Cloud project registered for Earth Engine.

When the cell finishes successfully, the notebook is ready to send processing requests to Earth Engine.


In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

# Cloud project registered for Earth Engine.
PROJECT_ID = "rstc2026-earth-engine"

# Authorise the Google account used in this Colab session.
ee.Authenticate()

# Initialise Earth Engine through the selected Cloud project.
ee.Initialize(project=PROJECT_ID)

print("Earth Engine connection successful.")

## 3. Define the study area and two local sampling sites

We use exactly the same Adventdalen AOI as in Block 1:

```text
west   = 15.75° E
south  = 78.16° N
east   = 16.05° E
north  = 78.24° N
```

The rectangle defines the area used later for spatial maps.

For time-series analysis we use two smaller local sites inside the AOI. Each point is surrounded by a **40 m radius buffer**. The buffer contains several Sentinel-2 pixels, so the local value is less sensitive to one anomalous pixel or a small geolocation difference.

The two locations are teaching examples rather than field plots.


In [ ]:
# Create the same rectangular Adventdalen AOI used in Block 1.
AOI = ee.Geometry.Rectangle(
    [15.75, 78.16, 16.05, 78.24],
    geodesic=False
)

# Define two local sample locations inside the AOI.
SITE_A_POINT = ee.Geometry.Point([15.90, 78.18])
SITE_B_POINT = ee.Geometry.Point([15.98, 78.20])

# Buffer each point by 40 m so the local value represents several nearby pixels.
SITE_RADIUS_M = 40
SITE_A = SITE_A_POINT.buffer(SITE_RADIUS_M)
SITE_B = SITE_B_POINT.buffer(SITE_RADIUS_M)

# Store both sampling areas together so we can loop over them later.
sites = {"Site A": SITE_A, "Site B": SITE_B}

### Display the AOI and sampling locations

This map is the first check of the analysis geometry.

Confirm that:

- both sample sites fall inside the Block 1 AOI;
- neither point is obviously located on water, infrastructure or another unsuitable surface;
- the two sites represent local parts of the Adventdalen landscape.

Later time-series plots are only meaningful if the sampling locations are sensible.


In [ ]:
Map = geemap.Map(
    center=[78.20, 15.90],
    zoom=10,
    basemap="Esri.WorldImagery"
)

Map.addLayer(AOI, {"color": "yellow"}, "Block 1 AOI")
Map.addLayer(SITE_A_POINT, {"color": "red"}, "Site A")
Map.addLayer(SITE_B_POINT, {"color": "cyan"}, "Site B")
Map

## 4. Load Sentinel-2 Level-2A and screen poor-quality pixels

This step builds the multi-year Sentinel-2 collection.

The code first:

1. selects images intersecting the Adventdalen AOI;
2. keeps acquisitions from May 2020 to September 2025;
3. removes only scenes with extremely high catalogue cloud cover.

The catalogue cloud percentage describes the whole Sentinel-2 tile, so it is not sufficient for a small AOI.

We therefore link each image to **Cloud Score+** and apply a pixel-level clear-sky threshold using `cs_cdf`.

We also remove pixels classified as **snow or ice** (`SCL = 11`). This is important in Arctic vegetation analysis because snow can create large spectral changes that are not vegetation change.

The result, `s2_clear`, is a Sentinel-2 collection in which poor-quality pixels are masked but valid pixels are retained.


In [ ]:
# Multi-year period used for the practical.
START_DATE = "2020-05-01"
END_DATE = "2025-10-01"
# Minimum Cloud Score+ clear-sky value retained.
CLEAR_THRESHOLD = 0.60

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
)

cloud_score = ee.ImageCollection(
    "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"
)

# Attach the cs_cdf quality band to the matching Sentinel-2 acquisitions.
s2 = s2.linkCollection(cloud_score, ["cs_cdf"])


def mask_quality(image):
    """Mask pixels that are not sufficiently clear or are classified as snow/ice."""
    # Keep pixels with a sufficiently high Cloud Score+ value.
    clear = image.select("cs_cdf").gte(CLEAR_THRESHOLD)
    # SCL class 11 represents snow or ice.
    not_snow = image.select("SCL").neq(11)
    return image.updateMask(clear.And(not_snow)).copyProperties(
        image, ["system:time_start"]
    )


s2_clear = s2.map(mask_quality)
print("Sentinel-2 collection prepared.")

## 5. Calculate NDVI, NDRE and NDMI

For every quality-filtered Sentinel-2 image we calculate three complementary indices.

### NDVI — Normalised Difference Vegetation Index

\[
NDVI = \frac{B8-B4}{B8+B4}
\]

NDVI uses red absorption and near-infrared reflectance and is our baseline metric for the green vegetation signal.

### NDRE — Normalised Difference Red Edge

\[
NDRE = \frac{B8A-B5}{B8A+B5}
\]

NDRE uses Sentinel-2 red-edge information. We include it to examine whether the red-edge response follows the same temporal pattern as NDVI.

### NDMI — Normalised Difference Moisture Index

\[
NDMI = \frac{B8A-B11}{B8A+B11}
\]

NDMI contrasts narrow NIR and SWIR reflectance. It is used here as moisture-sensitive environmental context rather than as another greenness index.

The Level-2A bands are first converted from stored integer values to surface reflectance by multiplying by `0.0001`.

Because NDRE and NDMI contain native 20 m bands, local extraction later uses a 20 m analysis scale.


In [ ]:
# Convert stored Sentinel-2 Level-2A values to surface reflectance.
SCALE = 0.0001


def add_indices(image):
    """Calculate NDVI, NDRE and NDMI for one quality-filtered Sentinel-2 image."""
    red = image.select("B4").multiply(SCALE)
    red_edge = image.select("B5").multiply(SCALE)
    nir = image.select("B8").multiply(SCALE)
    narrow_nir = image.select("B8A").multiply(SCALE)
    swir1 = image.select("B11").multiply(SCALE)

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    ndre = narrow_nir.subtract(red_edge).divide(
        narrow_nir.add(red_edge)
    ).rename("NDRE")
    ndmi = narrow_nir.subtract(swir1).divide(
        narrow_nir.add(swir1)
    ).rename("NDMI")

    return (
        image.addBands(ndvi)
        .addBands(ndre)
        .addBands(ndmi)
        .copyProperties(image, ["system:time_start"])
    )


s2_indices = s2_clear.map(add_indices)

## 6. Inspect a midsummer Sentinel-2 composite

Before extracting time-series values, inspect the spatial pattern of the data.

We select all valid observations from **July and August 2024** and calculate a temporal median.

Here, `median()` is calculated **pixel by pixel through time**. It does not calculate one median for the entire AOI.

The result is one midsummer composite image.

We display:

- RGB for visual interpretation;
- NDVI for the spatial vegetation pattern;
- the two sample sites.

Use this map to check the surface represented by each sample site before interpreting its time series.


In [ ]:
# Select July-August 2024 and calculate the temporal median at each pixel.
summer_2024 = (
    s2_indices
    .filterDate("2024-07-01", "2024-09-01")
    .median()
)

Map = geemap.Map(
    center=[78.20, 15.90],
    zoom=10,
    basemap="Esri.WorldImagery"
)

Map.addLayer(
    summer_2024,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2500},
    "Sentinel-2 RGB"
)
Map.addLayer(
    summer_2024.select("NDVI"),
    {"min": 0.0, "max": 0.7, "palette": ["f7f7f7", "d9f0a3", "78c679", "238443"]},
    "NDVI"
)
Map.addLayer(SITE_A_POINT, {"color": "red"}, "Site A")
Map.addLayer(SITE_B_POINT, {"color": "cyan"}, "Site B")
Map.addLayer(AOI, {"color": "yellow"}, "AOI", False)
Map

## 7. Extract the 2024 seasonal time series

We now convert the image collection into a small table of local observations.

First, we keep Sentinel-2 acquisitions from **May to September 2024**.

For every acquisition and every site, the function:

1. selects NDVI, NDRE and NDMI;
2. uses `reduceRegion()` to calculate the median value inside the 40 m buffer;
3. stores the acquisition date and site name;
4. returns one row of data for that satellite observation.

`getInfo()` transfers only this small table from Earth Engine to Colab. The full satellite images remain in Earth Engine.

If two Sentinel-2 granules contribute values on the same date, the final `groupby()` step combines them to one local median for that day.


In [ ]:
season_2024 = s2_indices.filterDate("2024-05-01", "2024-10-01")


def collection_to_dataframe(collection, geometry, site_name):
    """Extract one local Sentinel-2 time series and return it as a pandas table."""

    def image_to_feature(image):
        # Reduce the local 40 m sampling area to one median value per index.
        values = image.select(["NDVI", "NDRE", "NDMI"]).reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=geometry,
            scale=20,
            maxPixels=1e6
        )

        return (
            ee.Feature(None, values)
            .set("date", image.date().format("YYYY-MM-dd"))
            .set("site", site_name)
        )

    features = (
        ee.FeatureCollection(collection.map(image_to_feature))
        .filter(ee.Filter.notNull(["NDVI"]))
        .getInfo()["features"]
    )

    df = pd.DataFrame([f["properties"] for f in features])
    df["date"] = pd.to_datetime(df["date"])

    # If overlapping granules produce more than one value on the same date,
    # combine them to one local median for that day.
    return (
        df.groupby(["date", "site"], as_index=False)[["NDVI", "NDRE", "NDMI"]]
        .median()
        .sort_values("date")
    )


site_a_2024 = collection_to_dataframe(season_2024, SITE_A, "Site A")
site_b_2024 = collection_to_dataframe(season_2024, SITE_B, "Site B")
season_df = pd.concat([site_a_2024, site_b_2024], ignore_index=True)

season_df.head()

## 8. Compare the two local NDVI trajectories

This plot shows the retained Sentinel-2 NDVI observations for Site A and Site B during the 2024 growing season.

Look for differences in:

- the first seasonal increase in NDVI;
- the date and magnitude of the seasonal maximum;
- the timing of the late-season decline.

Each marker represents a retained satellite observation.

The line connects observations only to make the temporal sequence easier to read; it does not represent daily Sentinel-2 measurements.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for site_name in ["Site A", "Site B"]:
    subset = season_df[season_df["site"] == site_name]
    ax.plot(subset["date"], subset["NDVI"], marker="o", label=site_name)

ax.set(
    title="Local Sentinel-2 NDVI trajectories — 2024",
    xlabel="Date",
    ylabel="NDVI"
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 9. Examine relationships between the selected indices

For Site A we now compare NDVI, NDRE and NDMI using a **Spearman correlation matrix**.

Spearman correlation compares the ranking of observations and does not require a strictly linear relationship.

Use the matrix to ask:

- Do NDVI and NDRE rise and fall together?
- Does NDMI follow the same seasonal pattern?
- Does one index appear to add information that is not already represented by another?

This is a check for statistical similarity between indices. It is **not ecological validation**. Validation of biomass, chlorophyll or moisture would require independent field or environmental observations.


In [ ]:
site_a_metrics = site_a_2024[["NDVI", "NDRE", "NDMI"]].dropna()

display(
    site_a_metrics
    .corr(method="spearman")
    .round(2)
)

### Inspect NDVI and NDRE directly

A correlation coefficient reduces the relationship to one number.

The scatter plot shows the shape of the relationship.

Look for:

- an approximately straight relationship;
- curvature;
- clusters of observations;
- outliers.

This helps us see whether NDRE simply reproduces the NDVI pattern or behaves differently for some dates.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(site_a_metrics["NDVI"], site_a_metrics["NDRE"], s=45)
ax.set(
    title="NDVI versus NDRE — Site A, 2024",
    xlabel="NDVI",
    ylabel="NDRE"
)
ax.grid(alpha=0.3)
plt.show()

## 10. Compare midsummer NDVI between years

We now move from a seasonal analysis to an interannual comparison.

For every year from 2020 to 2025 the code:

1. selects the same July–August period;
2. creates a temporal median composite;
3. extracts local median NDVI for Site A and Site B.

Using the same seasonal window each year is important because it reduces the risk of comparing different parts of the growing season.

The resulting table contains one midsummer NDVI value for each site and year.

The following plot can be used to examine year-to-year variability and endpoint differences. Six years, however, are not enough to claim a robust long-term greening or browning trend.


In [ ]:
# This list will store one midsummer NDVI value for each site and year.
rows = []

# Repeat exactly the same July-August calculation for every year.
for year in range(2020, 2026):
    summer = (
        s2_indices
        .filterDate(f"{year}-07-01", f"{year}-09-01")
        .median()
    )

    for site_name, geometry in sites.items():
        ndvi = summer.select("NDVI").reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=geometry,
            scale=20,
            maxPixels=1e6
        ).get("NDVI").getInfo()

        rows.append({"year": year, "site": site_name, "NDVI": ndvi})

annual_df = pd.DataFrame(rows)
annual_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for site_name in ["Site A", "Site B"]:
    subset = annual_df[annual_df["site"] == site_name]
    ax.plot(subset["year"], subset["NDVI"], marker="o", label=site_name)

ax.set(
    title="Midsummer NDVI — 2020–2025",
    xlabel="Year",
    ylabel="July–August median NDVI"
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 11. Map spatial NDVI change across the full AOI

The two local time series tell us how selected locations behaved through time.

To examine **where** change occurred, we return to the full Block 1 AOI.

We create equivalent July–August median composites for 2020 and 2025 and calculate:

\[
\Delta NDVI = NDVI_{2025} - NDVI_{2020}
\]

Interpretation:

- positive values: higher midsummer NDVI in 2025;
- values near zero: little net difference;
- negative values: lower midsummer NDVI in 2025.

This is an **endpoint difference map**. It does not describe the intermediate years and it does not identify the ecological cause of the difference.


In [ ]:
# Create equivalent midsummer composites for the endpoint years.
summer_2020 = s2_indices.filterDate("2020-07-01", "2020-09-01").median()
summer_2025 = s2_indices.filterDate("2025-07-01", "2025-09-01").median()

# Subtract 2020 NDVI from 2025 NDVI pixel by pixel.
delta_ndvi = (
    summer_2025.select("NDVI")
    .subtract(summer_2020.select("NDVI"))
    .rename("delta_NDVI")
)

Map = geemap.Map(
    center=[78.20, 15.90],
    zoom=10,
    basemap="Esri.WorldImagery"
)

Map.addLayer(
    delta_ndvi,
    {
        "min": -0.15,
        "max": 0.15,
        "palette": ["8c510a", "f7f7f7", "01665e"]
    },
    "NDVI change: 2025 - 2020"
)
Map.addLayer(SITE_A_POINT, {"color": "red"}, "Site A")
Map.addLayer(SITE_B_POINT, {"color": "cyan"}, "Site B")
Map.addLayer(AOI, {"color": "yellow"}, "AOI", False)
Map

### Interpret the change map together with the time series

The map and the local plots answer different questions.

The map shows **where** the midsummer NDVI signal differs between 2020 and 2025.

The annual local series shows **how the two selected locations behaved across all six summers**.

A mapped difference may reflect vegetation change, but it can also be influenced by phenological timing, moisture, remaining snow, mixed pixels or unequal observation support.

For this reason, spatial change maps should be interpreted together with the underlying time series.


## 12. Reconstruct a seasonal NDVI curve for Site A

Sentinel-2 observations are irregular in time, and cloud screening creates additional gaps.

For the phenology demonstration we use Site A and:

1. keep the clear 2024 NDVI observations;
2. create a daily date axis between the first and last observation;
3. interpolate between observed dates;
4. smooth the interpolated series with a Savitzky–Golay filter.

The resulting daily curve is an **estimated seasonal trajectory**.

Interpolation and smoothing do not create new satellite observations. The original Sentinel-2 dates remain the observational support for the curve.


In [ ]:
pheno = site_a_2024[["date", "NDVI"]].dropna().sort_values("date")

if len(pheno) < 5:
    raise ValueError("Too few clear observations for a phenology demonstration.")

# Create a daily date axis between the first and last retained observation.
daily_dates = pd.date_range(pheno["date"].min(), pheno["date"].max(), freq="D")

daily_ndvi = (
    pheno.set_index("date")["NDVI"]
    .reindex(daily_dates)
    .interpolate(method="time")
)

# Smooth the interpolated trajectory; this creates an estimated seasonal curve.
window = min(21, len(daily_ndvi) if len(daily_ndvi) % 2 == 1 else len(daily_ndvi) - 1)

smooth_ndvi = savgol_filter(
    daily_ndvi.to_numpy(),
    window_length=window,
    polyorder=2
)

curve = pd.DataFrame({"date": daily_dates, "NDVI": smooth_ndvi})

## 13. Derive SOS, POS, EOS and LOS

We summarise the reconstructed seasonal curve using four phenology metrics.

**POS — Peak of Season**  
The date of maximum reconstructed NDVI.

**Seasonal amplitude**

\[
Amplitude = Peak - Baseline
\]

We define a relative threshold:

\[
Threshold = Baseline + 0.20 \times Amplitude
\]

**SOS — Start of Season**  
The first threshold crossing before the peak.

**EOS — End of Season**  
The final threshold crossing after the peak.

**LOS — Length of Season**

\[
LOS = EOS - SOS
\]

The 20% threshold is a transparent teaching definition. Different reconstruction methods or thresholds can produce different phenology dates.


In [ ]:
signal = curve["NDVI"].to_numpy()

# Locate the maximum of the reconstructed seasonal curve (POS).
peak_index = int(np.argmax(signal))
peak_value = float(signal[peak_index])

# Use the 10th percentile as a robust low-season baseline.
baseline = float(np.percentile(signal, 10))
amplitude = peak_value - baseline
# Set the phenology threshold at 20% of the seasonal amplitude.
threshold = baseline + 0.20 * amplitude

# First crossing before the peak = SOS.
sos_index = int(np.where(signal[:peak_index + 1] >= threshold)[0][0])

# Last value above the threshold after the peak = EOS.
eos_index = int(
    peak_index + np.where(signal[peak_index:] >= threshold)[0][-1]
)

sos = curve.loc[sos_index, "date"]
pos = curve.loc[peak_index, "date"]
eos = curve.loc[eos_index, "date"]
los = (eos - sos).days

print("Start of Season :", sos.date())
print("Peak of Season  :", pos.date())
print("End of Season   :", eos.date())
print("Length of Season:", los, "days")

## 14. Plot the observations and phenology metrics

The final figure shows both the original Sentinel-2 observations and the reconstructed seasonal curve.

This is important because the apparent precision of SOS, POS and EOS depends on the temporal support of the observations.

Check:

- whether observations exist before and after the seasonal maximum;
- whether there are long gaps;
- whether SOS or EOS falls inside a poorly observed interval.

The derived dates describe **land-surface phenology of the satellite signal** at Site A. They are not direct observations of individual plant phenological events.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.scatter(
    pheno["date"],
    pheno["NDVI"],
    s=45,
    label="Sentinel-2 observations",
    zorder=3
)
ax.plot(
    curve["date"],
    curve["NDVI"],
    linewidth=2,
    label="Reconstructed seasonal curve"
)
ax.axhline(threshold, linestyle="--", label="20% seasonal-amplitude threshold")
ax.axvline(sos, linestyle="--")
ax.axvline(pos, linestyle="--")
ax.axvline(eos, linestyle="--")

ax.text(sos, threshold, "  SOS", rotation=90, va="bottom")
ax.text(pos, peak_value, "  POS", rotation=90, va="top")
ax.text(eos, threshold, "  EOS", rotation=90, va="bottom")

ax.set(
    title="Land-surface phenology — Site A, 2024",
    xlabel="Date",
    ylabel="NDVI"
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

# What should you retain from this practical?

### Match spatial scale to the question

Use the full AOI when the question is:

> Where did the satellite vegetation signal change?

Use local sites when the question is:

> How did the vegetation signal change through time at this location?

### Match the metric to the process

NDVI, NDRE and NDMI use different spectral regions. Even when they correlate, they should not automatically be interpreted as the same environmental variable.

### Match the temporal design to the question

A seasonal trajectory, a year-to-year comparison and a five-year endpoint difference are different analyses.

### Inspect the observations behind every derived metric

A change value or phenology date is only as reliable as the satellite observations used to derive it.


# Student exercise

Choose one extension.

### Option A — compare phenology between the two sites

Repeat the phenology analysis for Site B and compare SOS, POS, EOS and LOS.

### Option B — compare two growing seasons

Extract the Site A seasonal NDVI observations for 2023 and compare them with 2024.

Consider both NDVI magnitude and seasonal timing.

### Option C — compare greenness and moisture response

Plot NDVI and NDMI through the 2024 growing season at Site A.

Ask:

> Do the two spectral signals change at the same time?

In every case, base the interpretation on the actual satellite observations before interpreting the derived metric.


# Technical references

- Sentinel-2 Level-2A Surface Reflectance Harmonized  
  https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Cloud Score+ S2 Harmonized  
  https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_CLOUD_SCORE_PLUS_V1_S2_HARMONIZED
- Sentinel-2 mission documentation  
  https://sentinels.copernicus.eu/copernicus/sentinel-2

The example sites are teaching locations. Ecological interpretation would require relevant independent field or environmental observations.